# Test Each Part of The Inference Pipeline

# Testing the Real World Cases with the BEST MODEL: BioBERT run_2!


In [ ]:
import os, json, torch, sys
from transformers import AutoModelForTokenClassification, AutoTokenizer


PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)

# Local imports 
from gcp_utils import download_from_gcs
from config import settings
from inference.v01.inference_utils import word_labels_to_spans, predict_word_level


In [3]:

# ------- Load labels and test data - LOCALLY -------
with open("../v01/data/biobert_splits/test.jsonl", "r") as f:
    test_data = []
    for line in f:
        test_data.append(line)

with open("../v01/data/id2label.json", "r") as f:
    id2label = json.load(f)
with open("../v01/data/label2id.json", "r") as f:
    label2id = json.load(f)
# ------------------------------------------------------

# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}


In [4]:
GCS_MODEL_PATH

'v02/runs/dmis-lab/biobert-base-cased-v1.1/run_2'

In [6]:
# CONFIG FOR LOADING FROM GCS 
VERSION = "v02"
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1" 
RUN_IDX = 2

GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{RUN_IDX}"
BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"

# Create a local directory path for the model
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"


In [9]:
BUCKET_NAME

'ner_training_data_results'

In [8]:
GCS_MODEL_PATH, LOCAL_MODEL_DIR

('v02/runs/dmis-lab/biobert-base-cased-v1.1/run_2',
 './downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_2')

In [10]:


# CONFIG FOR LOADING FROM GCS 
VERSION = "v02"
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1" 
RUN_IDX = 2

GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{RUN_IDX}"
BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"

# Create a local directory path for the model
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"

# Check if model already exists locally
if os.path.exists(LOCAL_MODEL_DIR) and os.path.isfile(os.path.join(LOCAL_MODEL_DIR, "config.json")):
    print(f"✅ Model found locally at {LOCAL_MODEL_DIR}")
    print("Skipping download from GCS.")
else:
    # Download the model directory from GCS if not found locally
    print(f"📥 Model not found locally. Downloading from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
    downloaded_path = download_from_gcs(
        gcs_path=GCS_MODEL_PATH,
        local_path=LOCAL_MODEL_DIR,
        bucket_name=BUCKET_NAME
    )
    if downloaded_path:
        print(f"✅ Download complete. Model saved to {LOCAL_MODEL_DIR}")

# Load the model
print(f"📂 Loading model from {LOCAL_MODEL_DIR}...")
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)

# Move to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
print(f"Using device: {device}")
model.to(device)
model.eval()

📥 Model not found locally. Downloading from gs://ner_training_data_results/v02/runs/dmis-lab/biobert-base-cased-v1.1/run_2...
❌ Failed to download v02/runs/dmis-lab/biobert-base-cased-v1.1/run_2: 403 GET https://storage.googleapis.com/storage/v1/b/ner_training_data_results/o?projection=noAcl&prefix=v02%2Fruns%2Fdmis-lab%2Fbiobert-base-cased-v1.1%2Frun_2%2F&prettyPrint=false: rferrazd.ai@gmail.com does not have serviceusage.services.use access to the Google Cloud project. Permission 'serviceusage.services.use' denied on resource (or it may not exist).
📂 Loading model from ./downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_2...


OSError: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory ./downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_2.

In [ ]:
# Import gold-standard cases
from real_world_cases import REAL_WORLD_CASES

for case in REAL_WORLD_CASES:
    text = case["text"]
    expected_entities = case.get("expected_entities", [])
    print(f"\n=== Case: {case['id']} ===")
    print(f"Text: {text}")

    tokens, token_labels, word_ids, words, word_labels = predict_word_level(
        text=text,
        model=model,
        tokenizer=tokenizer,
        id2label=id2label,
        device=device,
    )
    spans = word_labels_to_spans(words, word_labels)
  
    spans = [span_data for span_data in spans if span_data['label'] != 'O']
    print("Tokens:", tokens)
    print("Token-level labels:", token_labels)
    print("Word-level label indices:", word_labels)
    print("Predicted spans (Excluding 'O'):", spans)
    print("Expected entities:", expected_entities)
